<a href="https://colab.research.google.com/github/RitzKar/Ad_creator/blob/main/CrewAI_Tech16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install required pacakges


In [ ]:
%pip install --q crewai
%pip install --q -U duckduckgo-search
%pip install --q langchain_google_genai
%pip install langchain-openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 554.0 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.1/156.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.8/131.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.9/210.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.0/383.0 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.0/64.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Required Packages

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import OpenAI
from crewai import Agent, Task, Crew, Process

TypeError: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases

In [ ]:
from google.colab import userdata
# google_api_key = userdata.get('gemini_key')

### set up the LLM

In [ ]:
# # Set gemini pro as llm
# llm = ChatGoogleGenerativeAI(model="gemini-pro",
#                              verbose = True,
#                              temperature = 0.6,
#                              google_api_key=google_api_key)
llm = OpenAI(temperature=0.1, openai_api_key=userdata.get('open_ai_key'), model_name='o1-preview')

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = userdata.get('open_ai_key')

In [ ]:
llm

In [ ]:
llm.invoke("Hello how are you?")



### setup tools

In [ ]:
# Make sure to Install duckduckgo-search for this example:
# !pip install -U duckduckgo-search

from langchain.tools import DuckDuckGoSearchRun
search_tool = DuckDuckGoSearchRun()

In [ ]:
question_to_answer = "Conduct investment analysis on Airbnb"

**Sequential**


In [ ]:
# Define your agents with roles and goals
researcher = Agent(
  role='Senior Research Analyst',
  goal=f'Conduct comprehensive research to answer the question {question_to_answer}. With a particular focus on thinking about second order effects.',
  backstory="""You work at a leading analysis firm.
  Your expertise lies in analyzing complex issues.
  You have a knack for dissecting complex data and presenting
  actionable insights.""",
  verbose=True,
  allow_delegation=True,
  llm = llm,
  tools=[
        search_tool
      ]
)

critic = Agent(
  role='Critic',
  goal='Criticize output of research analysts to ensure the content is really insightful and not generic',
  backstory="""You are a senior analyst with an IQ of 125. You care alot about research and want to ensure insights are actionable""",
  verbose=True,
  allow_delegation=False,
  llm = llm,
  tools=[]
)

writer = Agent(
  role='Research Writer',
  goal=f'Craft a compelling research report to answer the question: {question_to_answer}',
  backstory="""You are a renowned Research Strategist, known for
  your insightful and engaging research.
  You transform complex concepts into compelling reports. Include a section on potential second order effects.""",
  verbose=True,
  allow_delegation=False,
  llm = llm,
  tools=[]
)


### Tasks to perform

In [ ]:
# Create tasks for your agents
task1 = Task(
  description=f"""Conduct a comprehensive research analysis to answer the question: {question_to_answer}""",
  agent=researcher,
  expected_output=f"Conduct a comprehensive research analysis to answer the question: {question_to_answer}"  # Add the expected output field
)

In [ ]:
# Create tasks for your agents
task2 = Task(
  description="""Critique the report and insights generated""",
  agent=critic,
  expected_output="A set of criticisms and suggested improvements"  # Add the expected output field
)

In [ ]:
task3 = Task(
  description=f"""Using the insights provided, develop a research analysis to answer the question: {question_to_answer}""",
  agent=writer,
  expected_output=f"A full research report of at least 4 paragraphs research analysis answering the question: {question_to_answer}." # Added expected output
)

### Create a Crew

In [ ]:
# Instantiate your crew with a sequential process
crew = Crew(
  agents=[researcher, critic, writer],
  tasks=[task1, task2, task3],
  verbose=2, # You can set it to 1 or 2 to different logging levels,
)

In [ ]:
crew

### Kickoff the crew - let the magic happen

In [ ]:
# Get your crew to work!
result = crew.kickoff()

# Heirarchical

In [ ]:
import os
from crewai import Agent, Task, Crew, Process

# Define your agents with roles and goals
researcher = Agent(
  role='Senior Research Analyst',
  goal=f'Conduct comprehensive research to answer the question {question_to_answer}. With a particular focus on thinking about second order effects.',
  backstory="""You work at a leading analysis firm.
  Your expertise lies in analyzing complex issues.
  You have a knack for dissecting complex data and presenting
  actionable insights.""",
  verbose=True,
  allow_delegation=True,
  llm = llm,
  tools=[
        search_tool
      ]
)

critic = Agent(
  role='Critic',
  goal='Criticize output of research analysts to ensure the content is really insightful and not generic',
  backstory="""You are a senior analyst with an IQ of 125. You care alot about research and want to ensure insights are actionable""",
  verbose=True,
  allow_delegation=False,
  llm = llm,
  tools=[]
)

writer = Agent(
  role='Research Writer',
  goal=f'Craft a compelling research report to answer the question: {question_to_answer}',
  backstory="""You are a renowned Research Strategist, known for
  your insightful and engaging research.
  You transform complex concepts into compelling reports. Include a section on potential second order effects.""",
  verbose=True,
  allow_delegation=False,
  llm = llm,
  tools=[]
)

# Define your task
task = Task(
    description=f"Conduct a comprehensive research analysis to answer the question: {question_to_answer}.",
    expected_output="A short concise report with insights for an investment analyst",
)

# Define the manager agent
manager = Agent(
    role="Project Manager",
    goal="Efficiently manage the crew and ensure high-quality task completion",
    backstory="You're an experienced project manager, skilled in overseeing complex projects and guiding teams to success. Your role is to coordinate the efforts of the crew members, ensuring that each task is completed on time and to the highest standard.",
    allow_delegation=True,
)

# Instantiate your crew with a custom manager
crew = Crew(
    agents=[researcher, writer, critic],
    tasks=[task],
    manager_agent=manager,
    process=Process.hierarchical,
)

# Start the crew's work
result = crew.kickoff()

In [ ]:
from IPython.display import display, Markdown
display(Markdown(result.raw))